# Entrega Parcial — Análise inicial da rede

**Disciplina:** Redes complexas — MO804/MC908 (Prof. Ruben Interian), Instituto de Computação, Unicamp
**Aluno:** André Luis Grotti Clemente

## A rede

A rede descreve as **relações comerciais, em 2018, no entorno de 35 estabelecimentos paulistas do setor de autopeças** (as *sementes*). Foi construída a partir dos registros administrativos da Nota Fiscal Eletrônica (NF-e) da Secretaria da Fazenda do Estado de São Paulo:

| Elemento | Definição |
|---|---|
| **Nó** | Um estabelecimento (CNPJ de 14 dígitos). O conjunto de nós é formado pelas 35 sementes e por todos os seus fornecedores e clientes diretos observados em notas de venda nos 12 meses anteriores à sua auditoria (a *camada 1*). |
| **Aresta** | Existe uma aresta entre dois nós do conjunto se houve **ao menos uma venda entre eles, em qualquer sentido, durante o ano de 2018** (NF-e modelo 55, notas de saída, CFOPs de venda). Trata-se portanto do **subgrafo induzido** pelo conjunto de nós: todas as arestas entre eles foram extraídas, e não só as que passam pelas sementes. |
| **Direção e peso** | O grafo é tratado como **simples, não direcionado e sem peso** nesta entrega. A direção (quem vende para quem) e o valor das operações foram preservados na fonte e serão usados nas entregas seguintes. |
| **Período** | Ano-calendário de 2018 (um único retrato temporal). |

Os dados são originais (extraídos diretamente da base da Sefaz-SP) e estão protegidos por sigilo fiscal (art. 198 do CTN). Por isso, o arquivo disponibilizado é uma **versão pública anonimizada**, gerada no ambiente interno por um procedimento que não compromete nenhuma das métricas calculadas aqui:

- **Identificadores:** o CNPJ de cada estabelecimento foi substituído por um *hash* criptográfico com chave secreta (HMAC-SHA256, truncado a 16 caracteres hexadecimais). A chave fica guardada no ambiente interno; sem ela, o hash não pode ser revertido nem recalculado a partir de uma lista de CNPJs.
- **Valores numéricos:** o valor negociado em cada aresta e o valor adicionado (VA) de cada nó foram **normalizados como frações do total da rede** (soma igual a 1), e o total não é divulgado. A transformação é um reescalonamento linear: preserva todas as razões e ordenações, e por isso não altera nenhuma medida topológica (itens 1 a 8, calculados sobre o grafo sem peso) nem as figuras, que usam apenas ordenações e proporções.
- **Atributos mantidos, por nó:** `camada` (0 = semente, 1 = vizinho), `uf` (`SP` ou `fora`), `direcao` (`semente`, `jusante` = cliente, `montante` = fornecedor, `ambas`), `tipo` (tipo de empresa, pela classe CNAE agregada em quatro categorias), `tem_va` e `va_norm`. **Por aresta:** `meses` (meses com venda em 2018, de 1 a 12) e `v_norm` (fração do fluxo total).

**Arquivo da rede (GraphML):** `rede_autopecas_sp_2018.graphml` — *[inserir link de acesso restrito]*

## Organização deste notebook

Cada um dos nove itens pedidos tem uma célula de texto com a definição da medida e a fórmula, seguindo os slides da disciplina (Aula 4 — *Vínculos fortes e fracos, coeficientes de clusterização*; Aula 8 — *Métricas de redes e centralidade*), e uma célula de código que a calcula com a biblioteca `igraph`.

**Notação.** $G = (V, E)$ é o grafo; $n = |V|$ é o número de vértices e $m = |E|$ o de arestas; $a_{ij} \in \{0, 1\}$ é a entrada da matriz de adjacência; $d(v_i)$ é o grau do vértice $v_i$.

In [ ]:
# --- Preparação: funciona no Jupyter (local ou servidor) e no Google Colab ---
import os, sys, subprocess
try:
    import igraph as ig
except ModuleNotFoundError:                                   # instala só se faltar
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'python-igraph'], check=True)
    import igraph as ig
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

ARQUIVO = 'rede_autopecas_sp_2018.graphml'
if 'google.colab' in sys.modules:                             # no Colab: monta o Drive e usa a pasta indicada
    from google.colab import drive; drive.mount('/content/drive')
    candidatos = ['/content/drive/MyDrive/NOME_DA_PASTA/publico/']   # <-- ajustar no Colab
else:                                                         # no Jupyter: a versão pública, ao lado do notebook ou no servidor
    candidatos = ['./publico/', './', '/home/candre/dados/rede/disciplina/publico/']
pasta = next((p for p in candidatos if os.path.exists(p + ARQUIVO)), None)
assert pasta, f'{ARQUIVO} não encontrado; coloque-o na mesma pasta do notebook ou em ./publico/'

g = ig.Graph.Read_GraphML(pasta + ARQUIVO)
if not g.is_simple():                                         # garante grafo simples, preservando atributos
    g.simplify(combine_edges={'meses': 'max', 'v_norm': 'sum'})
assert not g.is_directed()
assert 'tipo' in g.vs.attributes(), 'Este não é o GraphML da versão pública (faltam os atributos tipo/va_norm)'
camada = np.array(g.vs['camada'], dtype=int)                  # 0 = semente, 1 = vizinho
print(f'Grafo lido de {pasta + ARQUIVO}')
print(f'Atributos dos vértices: {g.vs.attributes()} | das arestas: {g.es.attributes()}')
print(f'Sementes: {(camada == 0).sum()} | vizinhos: {(camada == 1).sum():,}')

## 1. Número de vértices, número de arestas e grau médio

*Referência: Aula 8, slide 8 ("métricas globais triviais").*

- **Número de vértices** $n = |V|$ e **número de arestas** $m = |E|$ são as contagens básicas do grafo simples (sem laços, sem arestas múltiplas).
- O **grau** $d(v_i)$ de um vértice é o número de arestas incidentes a ele, isto é, o número de parceiros comerciais distintos do estabelecimento dentro da rede: $d(v_i) = \sum_j a_{ij}$.
- O **grau médio** é a média dos graus de todos os vértices. Como cada aresta contribui com uma unidade para o grau de cada uma de suas duas pontas, a soma dos graus é $2m$ e

$$\langle k \rangle = \frac{1}{n}\sum_{i=1}^{n} d(v_i) = \frac{2m}{n}.$$

Reportamos também o grau médio separado por camada, porque as sementes têm, por construção, todos os seus parceiros dentro do conjunto de nós, enquanto um vizinho só tem dentro dele a parte dos seus parceiros que também negocia com alguma semente.

In [ ]:
n, m = g.vcount(), g.ecount()
grau = np.array(g.degree())
print(f'Número de vértices (n): {n:,}')
print(f'Número de arestas  (m): {m:,}')
print(f'Grau médio <k> = 2m/n : {2 * m / n:.2f}   (conferência: média dos graus = {grau.mean():.2f})')
print(f'Grau mediano: {np.median(grau):.0f} | grau máximo: {grau.max():,} | grau mínimo: {grau.min()}')
print(f'Grau médio das sementes: {grau[camada == 0].mean():.1f} | dos vizinhos: {grau[camada == 1].mean():.1f}')

## 2. Coeficientes de clusterização global (as duas variantes)

*Referência: Aula 4, slides 24–36.*

Os coeficientes de clusterização medem o cumprimento do **princípio do fecho triádico**: se dois nós têm um vizinho em comum, há uma chance maior de serem adjacentes (Aula 4, slide 23). Em uma rede de fornecimento, o fecho triádico corresponde a dois parceiros de uma mesma empresa negociarem também entre si.

A base das duas variantes é o **coeficiente de clusterização local** $C_i$, definido no item 3. Os slides apresentam duas formas de agregar para a rede inteira, e **elas não são equivalentes**.

**Variante 1 — média dos coeficientes locais** (Aula 4, slide 30):

$$C = \frac{1}{n}\sum_{i} C_i .$$

Cada nó recebe o mesmo peso, tenha ele 2 ou 3.000 vizinhos. O slide alerta: *"o que fazer com nós de grau < 2? Estes nós precisam ser excluídos!"* — para eles o denominador $d(v_i)(d(v_i)-1)$ é zero e $C_i$ não é definido. Seguimos a orientação do slide e calculamos a média **só sobre os nós com grau $\geq 2$**. Como várias bibliotecas (NetworkX por padrão, e o `igraph` com `mode='zero'`) atribuem $C_i = 0$ a esses nós e os incluem na média, reportamos também esse valor para comparação, deixando claro qual é o adotado.

**Variante 2 — transitividade, ou razão de triplas** (Aula 4, slide 35):

$$C = \frac{\#\text{ triplas fechadas}}{\#\text{ triplas}} = \frac{3\,N_\triangle}{N_3} = \frac{3\sum_{k>j>i} a_{ij}a_{ik}a_{jk}}{\sum_{k>j>i}\left(a_{ij}a_{ik} + a_{ji}a_{jk} + a_{ki}a_{kj}\right)} ,$$

onde uma **tripla** é um conjunto de três nós conectados por ao menos duas arestas (um "caminho de comprimento 2", com centro no nó do meio), uma tripla é **fechada** quando a terceira aresta também existe, e $N_\triangle$ é o número de triângulos. O número de triplas fechadas é $3N_\triangle$ porque cada triângulo contém três triplas, uma centrada em cada vértice (slide 36). Nesta variante cada nó pesa pelo número de triplas que forma, $\binom{d(v_i)}{2}$, de modo que os nós de grau alto dominam o resultado.

Na célula seguinte calculamos as duas variantes de duas formas: pelas funções prontas do `igraph` e, para a transitividade, **pela contagem direta de triângulos e triplas**, como conferência da fórmula.

In [ ]:
# --- Variante 1: média dos coeficientes locais ---
C_local_zero = np.array(g.transitivity_local_undirected(mode='zero'))   # C_i = 0 se grau < 2
C_med_excl = C_local_zero[grau >= 2].mean()                              # adotado: exclui grau < 2 (slide 30)
C_med_incl = C_local_zero.mean()                                         # alternativa: inclui como 0
assert np.isclose(C_med_excl, g.transitivity_avglocal_undirected(mode='nan'))

# --- Variante 2: transitividade (razão de triplas) ---
C_trans = g.transitivity_undirected()
# conferência pela fórmula: 3 x triângulos / triplas
n_triangulos = int(round(np.sum(C_local_zero * grau * (grau - 1) / 2) / 3))   # T(v_i) = C_i·C(d_i,2); cada triângulo é contado em 3 nós
n_triplas = int(np.sum(grau * (grau - 1) / 2))        # cada nó de grau d é o centro de C(d,2) triplas
print(f'Variante 1 — média dos C_i, só nós com grau >= 2 (adotada): {C_med_excl:.4f}   '
      f'[{(grau >= 2).sum():,} nós considerados]')
print(f'Variante 1 — média dos C_i, grau < 2 contados como 0 (comparação): {C_med_incl:.4f}')
print(f'Variante 2 — transitividade (3 x triângulos / triplas): {C_trans:.4f}')
print(f'   conferência: triângulos = {n_triangulos:,} | triplas = {n_triplas:,} | '
      f'3 x {n_triangulos:,} / {n_triplas:,} = {3 * n_triangulos / n_triplas:.4f}')

**Como ler a diferença entre as duas variantes.** Quando a transitividade é bem menor que a média dos coeficientes locais, a rede tem **hubs com vizinhanças pouco conectadas**: um nó com milhares de parceiros forma milhões de triplas, quase todas abertas, e puxa a transitividade para baixo; os nós pequenos, cujos poucos parceiros negociam entre si, têm $C_i$ alto e puxam a média para cima. O item 3 mostra isso diretamente, pela queda de $C_i$ com o grau.

## 3. Coeficiente de clusterização de cada nó e sua distribuição

*Referência: Aula 4, slides 24–29.*

O **coeficiente de clusterização local** de um nó $v_i$ é o número de conexões existentes entre os seus vizinhos dividido pelo número máximo possível dessas conexões:

$$C_i = \frac{2\,T(v_i)}{d(v_i)\,\big(d(v_i)-1\big)} = \frac{\sum_{k>j} a_{ij}a_{ik}a_{jk}}{\sum_{k>j} a_{ij}a_{ik}},$$

onde $T(v_i)$ é o número de arestas entre vizinhos de $v_i$, que é igual ao número de triângulos que contêm $v_i$, e $d(v_i)(d(v_i)-1)/2$ é o número de pares de vizinhos.

Observações dos slides (26): $C_i$ varia de **0** (a vizinhança é uma *estrela*: nenhum vizinho ligado a outro) a **1** (a vizinhança é um *clique*: todos ligados entre si); e **se $d(v_i) = 0$ ou $1$, considera-se $C_i = 0$**, por convenção, já que não há pares de vizinhos a avaliar.

Para a empresa $v_i$, $C_i$ responde: *dos pares de parceiros comerciais de $v_i$, que fração também negocia entre si?*

A célula calcula $C_i$ para todos os nós, mostra a distribuição em histograma (toda a rede e só os nós com grau $\geq 2$) e, como complemento, a média de $C_i$ por faixa de grau, que explica a diferença entre as duas variantes globais do item 2.

In [ ]:
C_i = C_local_zero                                              # calculado no item 2, convenção C_i = 0 se grau < 2
print(pd.Series(C_i).describe(percentiles=[.1, .25, .5, .75, .9]).round(3).to_string())
print(f'\nNós com C_i = 0: {(C_i == 0).mean():.1%} | com C_i = 1: {(C_i == 1).mean():.1%}')

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(C_i, bins=40, color='steelblue')
ax[0].set(xlabel='$C_i$', ylabel='número de nós', title=f'Distribuição de $C_i$ — todos os {n:,} nós')
ax[1].hist(C_i[grau >= 2], bins=40, color='steelblue')
ax[1].set(xlabel='$C_i$', title=f'Só nós com grau ≥ 2 ({(grau >= 2).sum():,} nós)')
fig.tight_layout(); fig.savefig(pasta + 'fig_item3_clusterizacao_local.png', dpi=150); plt.show()

limites = [b for b in (0, 1, 5, 20, 100, 500) if b < grau.max()] + [int(grau.max())]
faixas = pd.cut(grau, limites)
print('\nC_i médio por faixa de grau:')
print(pd.DataFrame({'grau': faixas, 'C_i': C_i}).groupby('grau', observed=True)['C_i']
        .agg(nós='size', C_i_médio='mean').round(3).to_string())

## 4. Distância média entre os vértices

*Referência: Aula 8, slide 9.*

A **distância** $\ell(v_i, v_j)$ entre dois vértices é o comprimento (número de arestas) do **caminho mais curto** entre eles. A **distância média** é *"o valor médio entre todas as distâncias entre todos os pares de nós em um grafo conexo"*:

$$\langle \ell \rangle = \frac{1}{\binom{n}{2}} \sum_{i<j} \ell(v_i, v_j) = \frac{2}{n(n-1)}\sum_{i<j} \ell(v_i, v_j).$$

A definição supõe grafo conexo: se dois vértices estão em componentes diferentes, não existe caminho entre eles e a distância é infinita. Por isso, quando há mais de uma componente, a média é calculada **sobre os pares conectados** (ou, equivalentemente aqui, na maior componente). O item 8 mostra quantas componentes a rede tem; a célula calcula as duas versões e elas coincidem se a rede for conexa.

O cálculo exige uma busca em largura (BFS) a partir de cada vértice, custo $O(n \cdot m)$, viável para esta rede. Como complemento, mostramos a **distribuição das distâncias**: a fração de pares a 1, 2, 3, … passos, que descreve a rede melhor que a média sozinha.

In [ ]:
comp = g.connected_components()
gigante = comp.giant()
dist_media_conectados = g.average_path_length(unconn=True)      # média sobre os pares que têm caminho
dist_media_gigante = gigante.average_path_length()
print(f'Distância média entre os vértices (pares conectados): {dist_media_conectados:.4f}')
print(f'Distância média na maior componente ({gigante.vcount():,} nós): {dist_media_gigante:.4f}')

h = g.path_length_hist(directed=False)
dist = pd.Series({int(a): c for a, b, c in h.bins()}); dist = dist / dist.sum()
print('\nDistribuição das distâncias (fração dos pares conectados):')
print(dist.round(4).to_string())
print(f'Pares sem caminho (componentes diferentes): {h.unconnected:,}')

## 5. Diâmetro do grafo

*Referência: Aula 8, slide 9.*

O **diâmetro** é *"a maior distância entre dois nós diferentes"*, isto é, o maior entre todos os caminhos mais curtos:

$$D = \max_{i \neq j} \ell(v_i, v_j).$$

É uma medida de extremo: diz quão longe estão os dois nós mais afastados da rede, e costuma ser determinado por nós periféricos de grau baixo. Como a distância média, só é finito em um grafo conexo; em um grafo com várias componentes, reporta-se o diâmetro da maior componente (e, se for o caso, o maior diâmetro entre as componentes). A célula também identifica, pela distribuição do item 4, que fração dos pares está à distância máxima.

In [ ]:
diametro = gigante.diameter(directed=False)
print(f'Diâmetro (maior componente): {diametro}')
print(f'Fração dos pares conectados à distância máxima: {dist.get(diametro, 0):.6f}')
if len(comp) > 1:
    diam_comp = [g.subgraph(c).diameter() for c in comp]
    print(f'Maior diâmetro entre todas as componentes: {max(diam_comp)}')

## 6. Densidade

*Referência: Aula 8, slide 8.*

A **densidade** é *"número de arestas / número máximo possível de arestas"*. Em um grafo simples não direcionado com $n$ vértices, o máximo de arestas é o número de pares de vértices, $\binom{n}{2} = n(n-1)/2$; logo

$$\rho = \frac{m}{\binom{n}{2}} = \frac{2m}{n(n-1)}.$$

A densidade vale 1 no grafo completo e tende a 0 em redes grandes e esparsas. Ela se relaciona com o grau médio por $\rho = \langle k \rangle / (n-1)$.

In [ ]:
densidade = 2 * m / (n * (n - 1))
assert np.isclose(densidade, g.density())
print(f'Densidade = 2m / [n(n-1)] = 2 x {m:,} / ({n:,} x {n - 1:,}) = {densidade:.6f}')
print(f'Equivalente: <k>/(n-1) = {2 * m / n:.2f} / {n - 1:,} = {(2 * m / n) / (n - 1):.6f}')

## 7. Distribuição dos graus

*Referência: Aula 8 (grau e centralidade de grau); conceito de grau na Aula 4.*

A **distribuição dos graus** é a função $P(k)$ que dá a fração de vértices com grau exatamente $k$:

$$P(k) = \frac{n_k}{n}, \qquad n_k = \#\{\, i : d(v_i) = k \,\},$$

e descreve como a conectividade está repartida entre os nós. Ela é a assinatura mais usada para distinguir tipos de rede: em um grafo aleatório $P(k)$ é concentrada em torno de $\langle k \rangle$ (forma de Poisson); em muitas redes reais ela tem **cauda pesada** — poucos nós com grau muito acima da média (hubs) — e, em escala log-log, aproxima-se de uma reta, $P(k) \propto k^{-\gamma}$ (lei de potência).

Para visualizar a cauda usamos duas representações, ambas em escala log-log:

1. o **histograma** $n_k$ versus $k$;
2. a **distribuição acumulada complementar** (CCDF), $P(K \geq k) = \sum_{k' \geq k} P(k')$, que é mais estável na cauda, onde há poucos nós por valor de grau. Nela, uma lei de potência de expoente $\gamma$ aparece como reta de inclinação $-(\gamma - 1)$.

Mostramos a CCDF separada por camada: as sementes têm todos os seus parceiros dentro do recorte, os vizinhos não, e por isso as duas distribuições são diferentes por construção.

In [ ]:
k_vals, n_k = np.unique(grau, return_counts=True)
print('Distribuição dos graus (percentis):')
print(pd.Series(grau).describe(percentiles=[.1, .25, .5, .75, .9, .99]).round(1).to_string())
print(f'\nFração de nós com grau > 10 x <k> ({10 * 2 * m / n:.0f}): {(grau > 10 * 2 * m / n).mean():.2%}')

def ccdf(x):
    k, c = np.unique(x, return_counts=True)
    return k, 1 - (np.cumsum(c) - c) / c.sum()

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].loglog(k_vals, n_k, 'o', ms=3, color='steelblue')
ax[0].set(xlabel='grau $k$', ylabel='número de nós $n_k$', title='Distribuição dos graus (log-log)')
ax[1].loglog(*ccdf(grau), '-', color='black', label='todos os nós')
ax[1].loglog(*ccdf(grau[camada == 1]), '-', color='steelblue', label='vizinhos')
ax[1].loglog(*ccdf(grau[camada == 0]), 'o', ms=4, color='crimson', label='sementes')
ax[1].set(xlabel='grau $k$', ylabel='$P(K \\geq k)$', title='CCDF do grau, por camada'); ax[1].legend()
fig.tight_layout(); fig.savefig(pasta + 'fig_item7_distribuicao_graus.png', dpi=150); plt.show()

## 8. Número de componentes conexas

*Referência: Aula 8, slide 8.*

Uma **componente conexa** de um grafo não direcionado é um subconjunto maximal de vértices em que qualquer par está ligado por algum caminho. O número de componentes diz se a rede é um bloco único ou se há partes isoladas entre si. Em redes reais costuma existir uma **componente gigante** que contém a grande maioria dos nós, acompanhada de componentes pequenas.

A célula conta as componentes (algoritmo de busca em largura/profundidade) e informa o tamanho da maior e a fração de nós nela. Se houver mais de uma componente, plota a distribuição dos tamanhos (número de componentes por tamanho, em log-log).

In [ ]:
tamanhos = np.array(comp.sizes())
print(f'Número de componentes conexas: {len(comp)}')
print(f'Tamanho da maior componente: {tamanhos.max():,} nós ({tamanhos.max() / n:.2%} dos nós)')
if len(comp) > 1:
    print(f'Maiores componentes: {sorted(tamanhos.tolist(), reverse=True)[:10]}')
    t, c = np.unique(tamanhos, return_counts=True)
    fig, ax = plt.subplots(figsize=(5.5, 4))
    ax.loglog(t, c, 'o', color='steelblue')
    ax.set(xlabel='tamanho da componente (nós)', ylabel='número de componentes',
           title=f'Distribuição dos tamanhos — {len(comp)} componentes')
    fig.tight_layout(); fig.savefig(pasta + 'fig_item8_componentes.png', dpi=150); plt.show()
else:
    print('A rede é conexa: todos os nós estão em uma única componente; não há distribuição de tamanhos a plotar.')

## Resumo das métricas

In [ ]:
resumo = pd.Series({
    '1. vértices (n)': n,
    '1. arestas (m)': m,
    '1. grau médio <k> = 2m/n': round(2 * m / n, 2),
    '2. clusterização global — média dos C_i (grau >= 2)': round(C_med_excl, 4),
    '2. clusterização global — média dos C_i (grau < 2 como 0)': round(C_med_incl, 4),
    '2. clusterização global — transitividade': round(C_trans, 4),
    '3. C_i mediano': round(float(np.median(C_i)), 4),
    '4. distância média': round(dist_media_conectados, 3),
    '5. diâmetro': diametro,
    '6. densidade': round(densidade, 6),
    '7. grau máximo': int(grau.max()),
    '8. componentes': len(comp),
}, name='valor', dtype=object)
resumo.to_csv(pasta + 'metricas_resumo.csv')
print(resumo.to_string())

## 9. Visualização do grafo (complementar)

Com $m$ da ordem de meio milhão de arestas, desenhar todas as arestas produz uma mancha sem informação. A estratégia aqui é:

1. calcular o **layout de forças** (Fruchterman–Reingold) sobre um **esqueleto** do grafo — para cada nó, as arestas aos seus 3 vizinhos de maior grau, mais uma árvore geradora para manter a rede conexa — e desenhar só as arestas do esqueleto;
2. dimensionar cada nó por $\sqrt{d(v_i)}$ e destacar as sementes.

Em um layout de forças, a posição não tem significado métrico: nós próximos são nós que compartilham muitas ligações (diretas ou indiretas), e a distância no desenho não corresponde ao número de passos no grafo.

Os dois complementos ao final enriquecem esta figura com o tipo de empresa, o valor adicionado e os hubs, usando só os atributos do GraphML público.

In [ ]:
import random; random.seed(2026); ig.set_random_number_generator(random)

# esqueleto: 3 vizinhos de maior grau por nó + árvore geradora
viz_top = [(v, u) for v in range(n) for u in sorted(g.neighbors(v), key=lambda u: -grau[u])[:3]]
esq = ig.Graph(n=n, edges=viz_top + g.spanning_tree().get_edgelist(), directed=False).simplify()
xy = np.array(esq.layout_fruchterman_reingold(niter=1000, grid='auto').coords)
print(f'esqueleto: {esq.ecount():,} arestas desenhadas de {m:,}')

fig, ax = plt.subplots(figsize=(12, 12))
ax.add_collection(LineCollection(xy[np.array(esq.get_edgelist())], colors='#9a9a9a', linewidths=0.12, alpha=0.15))
viz = camada == 1
ordem = np.argsort(grau); ordem = ordem[viz[ordem]]            # pequenos primeiro, grandes por cima
ax.scatter(*xy[ordem].T, s=1 + 0.5 * np.sqrt(grau[ordem]), c='steelblue', lw=0, alpha=0.7, label=f'vizinhos ({viz.sum():,})')
ax.scatter(*xy[camada == 0].T, s=80, c='crimson', edgecolors='black', linewidths=0.6, zorder=5, label=f'sementes ({(camada == 0).sum()})')
ax.set_axis_off(); ax.set_aspect('equal')
ax.legend(loc='lower left', frameon=False, title='tamanho do nó ∝ √grau')
fig.savefig(pasta + 'fig_item9_rede.png', dpi=170, bbox_inches='tight'); plt.show()

## Complemento A — A rede com os hubs em destaque

A figura do item 9 mostra a forma da rede, mas não diz **quem são os nós mais conectados** nem **quanto eles pesam economicamente**. Este complemento acrescenta três informações a cada nó, todas presentes como atributos no GraphML público:

1. **Cor — o tipo de empresa** (`tipo`), pela classe CNAE principal do estabelecimento, agregada em quatro categorias: *indústria* (divisões 05 a 33), *comércio de autopeças* (classe 4530), *comércio e serviços gerais* (demais) e *sem CNAE em nenhuma fonte*. O CNAE foi obtido, nesta ordem de preferência, do cadastro estadual (CADESP, vigente em janeiro de 2018), do declarado na própria NF-e e, para estabelecimentos de outros estados, do cadastro nacional da Receita Federal (situação atual).
2. **Tamanho — o valor adicionado (VA) declarado em 2018** (`va_norm`), somado dos 12 meses da GIA (Guia de Informação e Apuração do ICMS), em módulo, normalizado como fração do VA total da rede e desenhado em escala de raiz quadrada, truncado no percentil 99 para que os maiores não esmaguem os demais. Só estabelecimentos paulistas do regime periódico entregam GIA; quem é do Simples Nacional ou de outro estado não tem VA observável (`tem_va = 0`) e recebe o tamanho mínimo.
3. **Contorno preto — os hubs**, definidos aqui como nós com grau $\geq 300$, mais de quatro vezes o grau médio da rede. As arestas do esqueleto que tocam um hub são desenhadas em tom mais escuro.

O layout é o mesmo tipo de desenho de forças do item 9 (Fruchterman–Reingold), mas calculado sobre um esqueleto formado pelas **três relações de maior valor de cada nó** (`v_norm`) mais uma árvore geradora, com a atração ponderada pelo logaritmo do valor relativo. Assim, as relações economicamente fortes é que aproximam os nós. Como em todo layout de forças, a posição não tem significado métrico: nós próximos compartilham muitas ligações, e a distância no desenho não é o número de passos no grafo.

**O que a figura permite ver.** Os hubs de maior grau, com exceção das próprias sementes, são em geral fornecedores de insumos de uso geral (material elétrico, informática, papelaria, ferragens, telecomunicações): empresas que vendem para todo o recorte sem que seus clientes tenham relação entre si. Eles têm grau enorme e VA pequeno em comparação com as indústrias, e explicam ao mesmo tempo a cauda da distribuição de graus (item 7) e a transitividade baixa (item 2): grau e importância econômica são coisas diferentes.

In [ ]:
# ---------- Complemento A: tipo de empresa, VA e hubs (só atributos do GraphML público) ----------
import random, matplotlib
from matplotlib.lines import Line2D
random.seed(2026); ig.set_random_number_generator(random)

# 1. atributos dos nós
tipo = np.array(g.vs['tipo'], dtype=object)
cnt = pd.Series(tipo).value_counts()
print('Nós por tipo de empresa:'); print(cnt.to_string())
va = np.array(g.vs['va_norm'], dtype=float); tem_va = np.array(g.vs['tem_va'], dtype=float) > 0
ref = np.quantile(va[tem_va], .99)
tam_va = np.where(tem_va, 3 + 160 * np.sqrt(np.clip(va / ref, 0, 1)), 1.5)
print(f'\nNós com VA observável: {tem_va.sum():,} de {n:,} ({tem_va.mean():.0%})')

# 2. esqueleto: 3 relações de maior valor de cada nó + árvore geradora; layout ponderado pelo valor
el = np.array(g.get_edgelist()); v = np.array(g.es['v_norm'], dtype=float)
w = pd.DataFrame({'u': el[:, 0], 'z': el[:, 1], 'v': v}).sort_values('v', ascending=False)
top = pd.concat([w.groupby('u').head(3), w.groupby('z').head(3)]).drop_duplicates(['u', 'z'])
esq = ig.Graph(n=n, edges=list(zip(top['u'], top['z'])) + g.spanning_tree().get_edgelist(), directed=False)
esq = esq.simplify()
vdict = {(a, b): x for a, b, x in zip(w['u'], w['z'], w['v'])}
pares = esq.get_edgelist()
ew = np.array([np.log1p(vdict.get((a, b), vdict.get((b, a), 0.0)) / np.median(v)) for a, b in pares])
ew = ew / ew.mean() + 0.05
xy2 = np.array(esq.layout_fruchterman_reingold(weights=ew.tolist(), niter=2000, grid='auto').coords)
print(f'Esqueleto: {esq.ecount():,} arestas desenhadas de {m:,}')

# 3. hubs e cores
cores = {'indústria': '#1f77b4', 'comércio de autopeças': '#ff7f0e', 'comércio e serviços gerais': '#d4d4d4',
         'sem CNAE em nenhuma fonte': '#2ca02c', 'semente': '#d62728'}
cor_set = np.array([matplotlib.colors.to_rgb(cores.get(c, '#999999')) for c in tipo])
hubs = grau >= 300
print(f'\nHubs (grau >= 300): {hubs.sum()} nós, dos quais {(hubs & (camada == 0)).sum()} sementes')
if hubs.sum():
    print('Hubs por tipo de empresa:'); print(pd.Series(tipo[hubs]).value_counts().to_string())

# 4. desenho
el_esq = np.array(pares); seg = xy2[el_esq]; toca_hub = hubs[el_esq].any(axis=1)
fig, ax = plt.subplots(figsize=(18, 18))
ax.add_collection(LineCollection(seg[~toca_hub], colors='#9a9a9a', linewidths=0.1, alpha=0.06))
ax.add_collection(LineCollection(seg[toca_hub], colors='#555555', linewidths=0.12, alpha=0.12))
o = np.argsort(tam_va); o = o[(camada[o] == 1) & ~hubs[o]]
ax.scatter(*xy2[o].T, s=tam_va[o], c=cor_set[o], lw=0, alpha=0.75)
h = hubs & (camada == 1)
ax.scatter(*xy2[h].T, s=np.maximum(tam_va[h], 40), c=cor_set[h], edgecolors='black', linewidths=1.0, zorder=4)
s_ = camada == 0
ax.scatter(*xy2[s_].T, s=np.maximum(tam_va[s_], 60), c='#d62728', edgecolors='black', linewidths=0.8, zorder=5)
ax.set_axis_off(); ax.set_aspect('equal')
ax.set_title(f'{n:,} estabelecimentos, 2018 — cor: tipo de empresa; tamanho ∝ √VA (fração do total, GIA); '
             f'contorno preto: hubs com grau ≥ 300 ({hubs.sum()} nós)', fontsize=12)
ax.legend(handles=[Line2D([], [], ls='', marker='o', color=c, label=l) for l, c in
    [(f'semente ({cnt.get("semente", 0)})', '#d62728'),
     (f'indústria ({cnt.get("indústria", 0):,})', '#1f77b4'),
     (f'comércio de autopeças ({cnt.get("comércio de autopeças", 0):,})', '#ff7f0e'),
     (f'comércio e serviços gerais ({cnt.get("comércio e serviços gerais", 0):,})', '#b0b0b0'),
     (f'sem CNAE em nenhuma fonte ({cnt.get("sem CNAE em nenhuma fonte", 0):,})', '#2ca02c'),
     (f'sem GIA em 2018 (Simples ou fora de SP): ponto mínimo ({(~tem_va).sum():,})', '#777777')]],
    loc='lower left', frameon=False, fontsize=10)
fig.savefig(pasta + 'fig_compA_rede_hubs_va.png', dpi=200, bbox_inches='tight'); plt.show()

## Complemento B — Zoom na região das sementes

*Depende da célula do Complemento A (mesmo layout, cores e tamanhos).*

As 35 sementes são as empresas auditadas em 2018 que deram origem à rede. Esta figura recorta a região do desenho anterior onde a maioria delas se concentra (entre os percentis 5 e 95 das coordenadas das sementes, com margem) e acrescenta duas camadas de informação:

- **Arestas semente–vizinho, em vermelho:** todas as ligações de cada semente com os nós do recorte, desenhadas por cima do esqueleto. Elas mostram a "estrela" de cada semente dentro do tecido de relações entre vizinhos. Como mostra a contagem impressa, essas arestas são uma fração pequena do total: a rede não é uma coleção de estrelas.
- **Arestas semente–semente, em preto:** vendas diretas entre duas empresas auditadas. Sementes ligadas entre si significam que uma auditada é fornecedora ou cliente de outra, o que é esperado entre fabricantes de autopeças que fornecem componentes uns aos outros. Apenas as arestas com as duas pontas dentro do recorte são desenhadas; as que levam a sementes fora dele aparecem na contagem impressa.

Cada semente recebe um **número de 1 a 35, que é o seu posto por grau** (1 = maior grau). É só um rótulo de ordenação, sem qualquer vínculo com a identidade da empresa. A célula também imprime, para registro, quantas sementes se ligam entre si, quantas componentes esse subgrafo das sementes tem, e a distribuição das **distâncias de grafo** entre pares de sementes, que é a medida objetiva de proximidade (ao contrário da distância no desenho).

In [ ]:
# ---------- Complemento B: zoom nas sementes (requer a célula do Complemento A) ----------
s_idx = np.where(camada == 0)[0]
el_g = np.array(g.get_edgelist())

# 1. relações entre sementes e distâncias de grafo entre elas
toca_sem = (camada[el_g] == 0).any(axis=1); ss = (camada[el_g] == 0).all(axis=1)
print(f'Arestas que tocam uma semente: {toca_sem.sum():,} de {m:,} ({toca_sem.mean():.1%})')
gss = g.subgraph(s_idx.tolist())
print(f'Arestas semente–semente: {ss.sum()} | sementes com ao menos uma: {np.unique(el_g[ss]).size} de {len(s_idx)} | '
      f'componentes do subgrafo das sementes: {len(gss.connected_components())}')
D = np.array(g.distances(source=s_idx.tolist(), target=s_idx.tolist()), dtype=float)
iu = np.triu_indices(len(s_idx), 1); Dp = D[iu][np.isfinite(D[iu])]
print('Pares de sementes por distância de grafo:', pd.Series(Dp.astype(int)).value_counts().sort_index().to_dict())
print(f'Distância média entre sementes: {Dp.mean():.2f} | máxima: {Dp.max():.0f}')

# 2. recorte do desenho
sx, sy = xy2[s_idx, 0], xy2[s_idx, 1]
x0, x1 = np.quantile(sx, [.05, .95]); y0, y1 = np.quantile(sy, [.05, .95])
mx, my = 0.25 * (x1 - x0), 0.25 * (y1 - y0)
x0, x1, y0, y1 = x0 - mx, x1 + mx, y0 - my, y1 + my
dentro = (xy2[:, 0] >= x0) & (xy2[:, 0] <= x1) & (xy2[:, 1] >= y0) & (xy2[:, 1] <= y1)
print(f'\nNós no recorte: {dentro.sum():,} de {n:,} | sementes no recorte: {(dentro & (camada == 0)).sum()} de {len(s_idx)}')
e_sem = toca_sem & ~ss & dentro[el_g].all(axis=1)                    # semente–vizinho, dentro do recorte
e_ss = ss & dentro[el_g].all(axis=1)                                  # semente–semente, dentro do recorte
e_esq = dentro[el_esq].all(axis=1) & ~(camada[el_esq] == 0).any(axis=1)
print(f'Arestas semente–semente com as duas pontas no recorte: {e_ss.sum()} de {ss.sum()}')

# 3. desenho
fig, ax = plt.subplots(figsize=(18, 18))
ax.add_collection(LineCollection(seg[e_esq], colors='#9a9a9a', linewidths=0.15, alpha=0.10))
ax.add_collection(LineCollection(xy2[el_g[e_sem]], colors='#d62728', linewidths=0.15, alpha=0.10))
ax.add_collection(LineCollection(xy2[el_g[e_ss]], colors='black', linewidths=1.2, alpha=0.9, zorder=5))
o = np.argsort(tam_va); o = o[dentro[o] & (camada[o] == 1) & ~hubs[o]]
ax.scatter(*xy2[o].T, s=tam_va[o] * 2.2, c=cor_set[o], lw=0, alpha=0.8)
h = dentro & hubs & (camada == 1)
ax.scatter(*xy2[h].T, s=np.maximum(tam_va[h] * 2.2, 90), c=cor_set[h], edgecolors='black', linewidths=1.2, zorder=4)
posto = pd.Series(grau[s_idx], index=s_idx).rank(ascending=False, method='first').astype(int)
for i in s_idx[dentro[s_idx]]:
    ax.scatter(*xy2[i], s=max(tam_va[i] * 2.2, 160), c='#d62728', edgecolors='black', linewidths=1.0, zorder=6)
    ax.annotate(str(posto[i]), xy2[i], ha='center', va='center', fontsize=8, color='white', weight='bold', zorder=7)
ax.set_xlim(x0, x1); ax.set_ylim(y0, y1); ax.set_axis_off(); ax.set_aspect('equal')
ax.set_title(f'Zoom na região das sementes: {(dentro & (camada == 0)).sum()} das {len(s_idx)} sementes e {dentro.sum():,} nós — '
             'número = posto da semente por grau; vermelho = arestas semente–vizinho; preto = arestas semente–semente; '
             'tamanho ∝ √VA; contorno preto = hub (grau ≥ 300)', fontsize=11)
fig.savefig(pasta + 'fig_compB_zoom_sementes.png', dpi=200, bbox_inches='tight'); plt.show()

## Observações sobre a construção da rede (limitações)

- **Fronteira do recorte.** Os nós foram selecionados a partir das sementes; os parceiros dos vizinhos que não negociam com nenhuma semente ficaram fora. O grau de cada vizinho dentro da rede é, portanto, um **limite inferior** do seu número de parceiros na NF-e, e a diferença entre o grau das sementes e o dos vizinhos é, em parte, efeito dessa construção.
- **Fronteira estadual.** A base registra notas emitidas em SP e notas de outros estados com destinatário em SP. Uma venda entre dois estabelecimentos de fora de SP não aparece, mesmo que ambos estejam na rede. O atributo `uf` permite avaliar a versão só com nós paulistas.
- **Amostragem por sementes.** A rede é o entorno de 35 empresas selecionadas (as auditadas em 2018), não uma amostra aleatória do setor; as sementes e os seus vizinhos tendem a ter grau acima da média do setor, e os resultados descrevem esse entorno.
- **Um retrato anual.** Uma aresta é qualquer venda em 2018, de uma nota única a um fornecimento contínuo. O peso (valor e número de meses) está disponível para as próximas entregas.